#### TF-IDF in NLP

TF-IDF stands for Term Frequency–Inverse Document Frequency.

It is a text vectorization technique used in NLP to convert text into numerical features.

Like Bag of Words (BoW), TF-IDF represents text numerically. However, there is one important difference:

Bag of Words: Counts how many times a word appears in a document.
TF-IDF: Gives a weight to a word based on its frequency in a document and how common or rare it is across the entire collection of documents.

<b>Transfer Frequency (TF) =</b> <span style="color:red;">Frequency of the word / Total no. of words in the sentence</span>

<b>Inverse Document Frequency (IDF) =</b> <span style="color:red;">log<sub>e</sub>(No. of sentenes / No. of sentences containing the word)</span>

<u><b>Note:</b> In case of "Binary TF-IDF":</u>
<br>Frequency of the word = 1 (if frequency >= 1)	
<br>Frequency of the word = 0 (if frequency = 0)

Example:

D1: <span style="color: blue;">He is a</span> good boy<br>
D2: <span style="color: blue;">She is a</span> good girl <span style="color: blue;">and a</span> good dancer<br>
D3: Boy <span style="color: blue;">and</span> girl <span style="color: blue;">are</span> good<br>
<table>
    <tr>
        <th><span style="color: blue;">Blue</span> = stop words</th>
    </tr>
</table>
<p>We can lower the case of all documents.</p>
<table>
    <tr>
        <th>Vocabulary</th>
        <th>Frequency</th>
    </tr>
    <tr>
        <td>good</td>
        <td>4</td>
    </tr>
    <tr>
        <td>boy</td>
        <td>2</td>
    </tr>
    <tr>
        <td>girl</td>
        <td>2</td>
    </tr>
    <tr>
        <td>dancer</td>
        <td>1</td>
    </tr>
</table>
<b>Note:</b> TFIDFVectorizer sorts the vocabulary based on alphabatical order not frequency order

<p><u>Transfer Frequency (TF):</u></p>
<table>
    <tr>
        <th></th>
        <th><span style="color:red;">D1</span></th>
        <th><span style="color:red;">D2</span></th>
        <th><span style="color:red;">D3</span></th>
    </tr>
    <tr>
        <td>boy</td>
        <td>1/2</td>
        <td>0</td>
        <td>1/3</td>
    </tr>
    <tr>
        <td>dancer</td>
        <td>0</td>
        <td>1/4</td>
        <td>0</td>
    </tr>
    <tr>
        <td>girl</td>
        <td>0</td>
        <td>1/4</td>
        <td>1/3</td>
    </tr>
    <tr>
        <td>good</td>
        <td>1/2</td>
        <td>2/4 = 1/2</td>
        <td>1/3</td>
    </tr>
</table>

<p><u>Inverse Document Frequency (IDF):</u></p>
<table>
    <tr>
        <td><span style="color:red;">boy</span></td>
        <td>log<sub>e</sub>(3/2)</td>
    </tr>
    <tr>
        <td><span style="color:red;">dancer</span></td>
        <td>log<sub>e</sub>(3/1)</td>
    </tr>
    <tr>
        <td><span style="color:red;">girl</span></td>
        <td>log<sub>e</sub>(3/2)</td>
    </tr>
    <tr>
        <td><span style="color:red;">good</span></td>
        <td>log<sub>e</sub>(3/3) = 0</td>
    </tr>
</table>

<p><u>TF-IDF Vectorization:</u></p>
<table>
    <tr>
        <th></th>
        <th><span style="color:red;">boy</span></th>
        <th><span style="color:red;">dancer</span></th>
        <th><span style="color:red;">girl</span></th>
        <th><span style="color:red;">good</span></th>
    </tr>
    <tr>
        <td><span style="color:red;">D1</span></td>
        <td>1/2 * log<sub>e</sub>(3/2) = 0.088</td>
        <td>0</td>
        <td>0</td>
        <td>0</td>
    </tr>
    <tr>
        <td><span style="color:red;">D2</span></td>
        <td>0</td>
        <td>1/4 * log<sub>e</sub>(3) = 0.119</td>
        <td>1/4 * log<sub>e</sub>(3/2) = 0.044</td>
        <td>0</td>
    </tr>
    <tr>
        <td><span style="color:red;">D3</span></td>
        <td>1/3 * log<sub>e</sub>(3/2) = 0.058</td>
        <td>0</td>
        <td>1/3 * log<sub>e</sub>(3/2) = 0.058</td>
        <td>0</td>
    </tr>
</table>

In [2]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from nltk import pos_tag
import pandas as pd

messages = pd.read_csv("dataset/spamHam.csv", sep=",", 
                      names=["label", "message"], encoding="utf-8")

# Convert Penn Treebank POS tags to WordNet POS tags
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

filtered_documents=[]
for i in range(0, len(messages)) :
    # print(messages['message'][i])
    # review = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    tokens = word_tokenize(messages['message'][i])
    
    # Remove stop words before POS tagging
    filtered_tokens=[]
    for token in tokens:
        if token.isalnum() and token.lower() not in stop_words:
            filtered_tokens.append(token.lower())
    
    # POS tagging
    tagged_tokens = pos_tag(filtered_tokens)

    # Lemmatization using converted POS tags
    lemmatized_tokens = []

    for token, tag in tagged_tokens:
        wordnet_pos = get_wordnet_pos(tag)
        lemmatized_token = lemmatizer.lemmatize(token, wordnet_pos)
        lemmatized_tokens.append(lemmatized_token)
        
    # Store POS tags for lemmatized words
    filtered_documents.append(" ".join(lemmatized_tokens))

filtered_documents   

['hey free talk even rush free entire day today',
 'alert tax refund ready submit bank detail receive',
 'hey message get',
 'notice offer get 70 next purchase shop offer subject term',
 'exclusive mobile number shop voucher call redeem check detail today',
 'hey get home safely talk soon',
 'hey could check whether office open let know',
 'exclusive urgent payment require prevent service disconnect offer subject term',
 'alert account block unless confirm detail check detail today',
 'let know hi still meet lunch today rush',
 'package wait pay small fee use link release',
 'notice get instant approval personal loan apply offer subject term',
 'quick question time movie start tonight let know',
 'alert stream account hold update payment information offer subject term',
 'hey cooking dinner would like',
 'hey thanks receive photo rush',
 'let know read book recommend',
 'wallet credit bonus click withdraw',
 'hi would like tea coffee let know',
 'quick question thanks help assignment l

#### TfidfVectorizer

```
class sklearn.feature_extraction.text.TfidfVectorizer(*, input='content', encoding='utf-8', decode_error='strict', strip_accents=None, lowercase=True, preprocessor=None, tokenizer=None, analyzer='word', stop_words=None, token_pattern='(?u)\\b\\w\\w+\\b', ngram_range=(1, 1), max_df=1.0, min_df=1, max_features=None, vocabulary=None, binary=False, dtype=<class 'numpy.float64'>, norm='l2', use_idf=True, smooth_idf=True, sublinear_tf=False)
```

Link: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [15]:
# Create Bag Of Words model
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidfVec = TfidfVectorizer(max_features=2500, binary=False)

X = tfidfVec.fit_transform(filtered_documents).toarray()

np.set_printoptions(edgeitems=30, linewidth=100000)
print(X)

# for arr in X:
#     for element in arr:
#         print(element, end="                   ")
#     print()

[[0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         ... 0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.25413419 0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.34382107 0.         0.         0.         0.         0.         0.         .

In [21]:
# Create Bag Of Words model
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidfVec = TfidfVectorizer(max_features=2500, binary=True, ngram_range=(1,2))

X = tfidfVec.fit_transform(filtered_documents).toarray()

np.set_printoptions(edgeitems=30, linewidth=100000)
print(X)

# for arr in X:
#     for element in arr:
#         print(element, end="                   ")
#     print()

[[0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         ... 0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         .

In [22]:
tfidfVec.vocabulary_

{'hey': np.int64(486),
 'free': np.int64(437),
 'talk': np.int64(1161),
 'even': np.int64(346),
 'rush': np.int64(1065),
 'entire': np.int64(336),
 'day': np.int64(270),
 'today': np.int64(1193),
 'hey free': np.int64(497),
 'free talk': np.int64(444),
 'talk even': np.int64(1162),
 'even rush': np.int64(348),
 'rush free': np.int64(1066),
 'free entire': np.int64(439),
 'entire day': np.int64(337),
 'day today': np.int64(274),
 'alert': np.int64(31),
 'tax': np.int64(1170),
 'refund': np.int64(1029),
 'ready': np.int64(1007),
 'submit': np.int64(1149),
 'bank': np.int64(111),
 'detail': np.int64(291),
 'receive': np.int64(1009),
 'alert tax': np.int64(63),
 'tax refund': np.int64(1171),
 'refund ready': np.int64(1031),
 'ready submit': np.int64(1008),
 'submit bank': np.int64(1150),
 'bank detail': np.int64(113),
 'detail receive': np.int64(295),
 'message': np.int64(740),
 'get': np.int64(451),
 'hey message': np.int64(506),
 'message get': np.int64(741),
 'notice': np.int64(792),
 '